# Lab 2: Molecular Dynamics of Proteins (OpenMM)
### University of California, Berkeley - Spring 2026 - ME 120/292A


## Overview

### Purpose
The purpose of this lab is to introduce **Molecular Dynamics (MD)** as a computational tool for studying biomolecular mechanics at the atomic and molecular scale. You will connect the MD workflow steps from lecture (forces from a potential energy function, time integration, temperature control, and equilibrium vs. production) to hands-on simulations using **OpenMM**.

In the OpenMM portion of this lab, you will run short MD simulations of simple toy polypeptides and explore how temperature (and optionally the environment: vacuum vs solvent) influences their motion and qualitative conformational behavior. Because these runs are short and the systems are simplified, you should interpret results as relaxation and fluctuations under the chosen model and conditions, not guaranteed native folding.

### Environment
In this lab we will work with the following tools:
- [OpenMM](https://openmm.org/) for running MD simulations in Python.
- [MDAnalysis](https://www.mdanalysis.org/) for reading trajectory files.
- [py3Dmol](https://3dmol.csb.pitt.edu/) for interactive trajectory visualization inside Jupyter.

### Objectives
By the end of this lab, you should be able to:
- Explain how forces are computed from a potential energy function U(x) and integrated in time to produce a trajectory.
- Run a basic OpenMM workflow: system setup -> minimization -> (optional) equilibration -> production.
- Compare trajectories across conditions (at least temperature; optionally vacuum vs solvent) and make cautious qualitative conclusions.

### Submission
Submit a short PDF report named:
- `lab2_MD_{firstname}_{lastname}.pdf`

Submit to bCourses Assignment.

## Setup

### Files
- Starter files: `Lab2-MD_sim/`
- Where to run: DataHub or local

Download the lab directory from bCourses as a `.zip` archive. **Extract** it into your `ME120/` or `ME292A/` course folder (DataHub or local). Run the notebook from inside the extracted lab folder so relative paths like `data/` and `results/` work correctly.

You should have at least:
- `MD.ipynb`
- `md1.py`
- `environment.yml`
- `data/` (one or more `.pdb` files)

### Dependencies

Create the environment from YAML:
```bash
conda env create -f lab2-environment.yml
```

Create environment manually:
```bash
conda create -n openmm python=3.8
conda activate openmm
conda install ipykernel openmm mdanalysis py3dmol ffmpeg
```

Activate:
```bash
conda activate openmm
```

(Optional) Register the environment as a Jupyter kernel:
```bash
python -m ipykernel install --user --name openmm --display-name openmm
```

In [1]:
from md1 import simulate_apple_fall, simulate_three_particles
from IPython.display import Video, Image

ModuleNotFoundError: No module named 'md1'

## 1. MD foundations: Newton's laws

Newton's 2nd law connects the kinematics (movements) of a body with its mechanics (total force acting on it) and defines the dynamic evolution of its position: 

From Lecture 2:
$$
m_i \frac{d^2 x_i}{dt^2} = \sum^{N}_{j=1} F_{ij}, \qquad i = 1,\dots,N
$$

Alternatively:
$$
m_i \frac{d^2 \mathbf{r}_i}{dt^2} = \sum_{j} \mathbf{F}_{ij}, \qquad i = 1,\dots,N.
$$

The forces come from a potential energy (a force field). In molecular mechanics we typically write:

$$\mathbf{F}_i = -\nabla_{\mathbf{r}_i} U(\mathbf{r}_1,\dots,\mathbf{r}_N).$$

where $m$ is the mass, $r$ is the position, $F$ is the force and $U(r)$ is the potential energy, which depends only on the position of the body.

If one knows the forces acting upon the body, one can find the position of the body at any moment $r(t)$, or predict its dynamics. This can be done by solving Newton's equation of motion. It is a second order ODE that can be solved analytically for a few simple cases: constant force, harmonic oscillator, periodic force, drag force, etc.

However, a more general approach is to use computers in order to solve the ODE numerically.

## 2. Time integration with particle systems

There are [many methods](https://en.wikipedia.org/wiki/Numerical_methods_for_ordinary_differential_equations) for solving ODEs. The second order ODE is transformed to the system of two first order ODEs as follows:

$$\frac{dr(t)}{dt} = v(t)$$

$$m\frac{dv(t)}{dt} = F(t)$$

We use a finite difference approximation that comes to a simple forward Euler Algorithm: 

$$ v_{n+1} = v_n + \frac{F_n}{m} dt$$

$$ r_{n+1} = r_n + v_{n+1} dt$$

Here we discretize time t with time step $dt$, so $t_{n+1} = t_n + dt$, and $r_{n} = r(t_n)$, $v_{n} = v(t_n)$, where $n$ is the timestep number. Using this method, computing dynamics is straightforward.

### Projectile motion warm-up

We want to know the dynamics of a green apple ($m = 0.3$ kg) tossed horizontally at 10 cm/s speed from the top of the Toronto CN Tower (553 m) for the first 10 seconds.

In [ ]:
simulate_apple_fall(
     total_time=10, 
     mass=0.3, 
     initial_velocity=0.1, 
     height=553, 
     timestep=0.05,
 )

Video('results/apple_fall.mp4')

# # Use if ffmpeg is not installed, Make sure to change md1.py to save the animation as gif
# # Image('results/apple_fall.gif')

When a closed system of particles interact through pairwise potentials, the force on each particle $i$ depends on its position with respect to every other particle $j$:

$$m_i\frac{d^2\mathbf{r}_i(t)}{dt^2} = \sum_j \mathbf{F}_{ij}(t) = -\sum_j \nabla_{\mathbf{r}_i} \, U\big(r_{ij}(t)\big)$$

where we define the **vector separation** and **scalar distance** as:

$$\mathbf{r}_{ij} := \mathbf{r}_i - \mathbf{r}_j, \qquad r_{ij} := \|\mathbf{r}_{ij}\|.$$

In coordinates, if $\mathbf{r}_i=(x_i,y_i,z_i)$ and $\mathbf{r}_j=(x_j,y_j,z_j)$, then:

$$r_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2 + (z_i - z_j)^2},$$

with $i,j \in \{1,\dots,N\}$.


### Three-body system with Hooke's law

We want to know the dynamics of 3 particles $m = 1$ kg connected to each other with invisible springs with $K_s = 5$ N/m, and $r_0 = 1$ m initially located at (0, 2), (2, 0) and (-1, 0) on the 2D plane for the first 10 seconds of their motion.

**Hint:**
The pairwise potential is (Hooke's Law): $$U(r_{ij}) = \frac{K_s}{2}(r_{ij} - r_0)^2$$

The negative gradient of the potential is a force from the $j$-th particle on the $i$-th particle:

$$\mathbf{F}_{ij} = -\nabla_{\mathbf{r}_i} U(r_{ij}).$$

Using $\mathbf{r}_{ij} := \mathbf{r}_i - \mathbf{r}_j$ and $r_{ij} := \|\mathbf{r}_{ij}\|$, we have:

$$\nabla_{\mathbf{r}_i} r_{ij} = \frac{\mathbf{r}_{ij}}{r_{ij}}$$

so:

$$\mathbf{F}_{ij} = - K_s (r_{ij} - r_0) \, \frac{\mathbf{r}_{ij}}{r_{ij}}.$$


In [ ]:
simulate_three_particles(
    total_time=10, mass=1.0, ks=5, r0=1.0, timestep=0.05
 )

Video('results/3particles.mp4')

# # Use if ffmpeg is not installed, Make sure to change md1.py to save the animation as gif
# # Image('results/3particles.gif')

## 3. Proteins: Structure and Function



While we now have a basic knowledge of the purpose and methodology of simulations, we still need to understand what proteins are and why they are important.

[Protein structure](https://en.wikipedia.org/wiki/Protein_structure) is the three-dimensional arrangement of atoms in a protein, which is a chain of amino acids. Proteins are polymers, specifically polypeptides, formed by combined sequences of 20 different types of amino acids, which are the monomers of this polymer. A single amino acid monomer may also be called a residue, indicating a repeating unit of a polymer. To be able to perform their biological function, proteins fold into one or more specific spatial conformations driven by a number of non-covalent interactions such as:

- Hydrogen bonding 
- Ionic interactions 
- Van der Waals forces
- Hydrophobic effects

To understand the functions of proteins at a molecular level, it is often necessary to determine their three-dimensional structure experimentally using techniques such as X-ray crystallography, NMR spectroscopy, and others.

From lecture and the biology crash course you learned the following:

### Levels of structure

**Primary structure** of a protein refers to the sequence of amino acids in the polypeptide chain.

**Secondary structure** refers to highly regular local sub-structures of the actual polypeptide backbone chain. There are two main types of secondary structure: the alpha-helix and the beta-strand or beta-sheets.

**Tertiary structure** refers to the three-dimensional structure of monomeric and multimeric protein molecules. The alpha-helixes and beta-sheets are folded into a compact globular structure. 

**Quaternary structure** is the three-dimensional structure consisting of two or more individual polypeptide chains (subunits) that operate as a single functional unit (multimer).


### Functions
Proteins often can satisfy many standard functions in biological systems. 
- *Antibodies* - bind to specific foreign particles, ex: IgG 
- *Enzymes* - speed up chemical reactions, ex: Lysozyme
- *Messengers* - transmit signals, ex: Growth hormone 
- *Structural components* - support for cells, ex: Tubulin
- *Transport/storage* - bind and carry small molecules, ex: Hemoglobin


## 4. Molecular Mechanics and Force Fields


Since we now know what proteins are and why these molecular machines are important, we consider the method to model them. The basic idea is to create the same kind of approach as we used in the 3-body simulation. Now, our system consists of thousands of particles (atoms of the protein plus atoms of surrounding water) and they all are connected via a complex potential energy function.

An all-atom potential energy function $U$ is usually given by the sum of the bonded terms ($U_b$) and non-bonded terms ($U_{nb}$), i.e.

$$U = U_{b} + U_{nb},$$

where the bonded potential includes the harmonic (covalent) bond part, the harmonic angle and
the two types of torsion (dihedral) angles: proper and improper. As it can be seen, these functions are mostly harmonic potentials:

$$U_{b} = \sum_{bonds}\frac{1}{2}K_b(b-b_0)^2 + \sum_{angles}K_{\theta}(\theta-\theta_0)^2 + \sum_{dihedrals}K_{\phi}(1-\cos(n\phi - \phi_0)) + \sum_{impropers}K_{\psi}(\psi-\psi_0)^2$$

For example, $b$ and $\theta$ represent the distance between two atoms and the angle between two
adjacent bonds; $\phi$ and $\psi$ are dihedral (torsion) angles. These can be evaluated for all the
atoms from their current positions. Also, $K_b$, $K_\theta$, $K_\phi$, and $K_\psi$ are the spring constants, associated
with bond vibrations, bending of bond angles, and conformational fluctuations in dihedral and
improper angles around some equilibrium values $b_0$, $\theta_0$, $\phi_0$, and $\psi_0$, respectively.

The non-bonded part of the potential energy function is represented by the electrostatic and van der Waals potentials, i.e.

$$U_{nb} = \sum_{i,j}\left(\frac{q_{i}q_{j}}{4\pi\varepsilon_{0}\varepsilon \, r_{ij}} + \varepsilon_{ij}\left[\left(\frac{\sigma^{min}_{ij}}{r_{ij}}\right)^{12}-2\left(\frac{\sigma^{min}_{ij}}{r_{ij}}\right)^{6}\right]\right)$$

where $r_{ij}$ is the distance between two interacting atoms, $q_i$ and $q_j$ are their electric charges; $\varepsilon$ and
$\varepsilon_0$ are electric and dielectric constant; $\varepsilon_{ij} = \sqrt{\varepsilon_i\varepsilon_j}$ and
$\sigma_{ij} = \frac{\sigma_i + \sigma_j}{2}$ are van der Waals parameters for atoms $i$ and $j$.

**Importantly, each force field has its own set of parameters, which are different for different types of atoms.**


## 5. Run protein MD in OpenMM


[**Molecular dynamics (MD)**](https://en.wikipedia.org/wiki/Molecular_dynamics) is a computer simulation method for studying the physical movements of atoms and molecules, i.e. their dynamical evolution. 

In the most common version, the trajectories of atoms and molecules are determined by numerically solving Newton's equations of motion for a system of interacting particles, where forces between the particles and their potential energies are often calculated using  molecular mechanics force fields. 



Now with all that intellectual equipment, we can start running legit Molecular Dynamics simulations. All we need is an initial structure of the protein and software that computes its dynamics efficiently.

### Workflow

1. Load initial coordinates of protein atoms (from a `.pdb` file)
2. Choose force field parameters.
3. Choose parameters of the experiment: temperature, pressure, box size, solvation, boundary conditions
4. Choose integrator: algorithm for solving equation of motion
5. Run simulation, save coordinates time to time (to a trajectory file, typically `.dcd`).
6. Visualize the trajectory
7. Perform the analysis

Packages used:

* **py3Dmol**: py3Dmol is a Jupyter-friendly wrapper around 3Dmol.js for interactive molecular graphics (cartoons, sticks, spheres) rendered with WebGL.

* __MDAnalysis__: MDAnalysis is an object-oriented Python library to analyze trajectories from molecular dynamics (MD) simulations in many popular formats. It can write most of these formats, too, together with atom selections suitable for visualization or native analysis tools.

* __OpenMM__: Openmm consists of two parts: One is a set of libraries that lets programmers easily add molecular simulation features to their programs and the other part is an "application layer" that exposes those features to end users who just want to run simulations

### Example: short peptide dynamics (polyalanine)

In this example, you will run a short MD simulation of a fully extended polyalanine peptide (`polyALA.pdb`) in an *in vacuo* environment.

Because DataHub has limited compute resources and session availability, the default run is intentionally short. That is enough to:

1. Practice the **MD workflow** (setup → minimize → optional equilibration → production).
2. Save a trajectory file (`.dcd`) in `results/`.
3. Visualize the trajectory with `py3dmol`.

> Note: These are toy systems and short simulations. Do not expect a fully folded structure. Focus on getting the workflow running and making qualitative observations as you change parameters (temperature, environment).


In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import MDAnalysis as mda
from sys import stdout
import pathlib
import py3Dmol
from md1 import visualize


c:\Users\kentc\miniconda3\envs\openmm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load the Protein (polypeptide) PDB file

Run the next cell and **choose ONE** peptide (`polyALA`, `polyGLY`, or `polyGV`) by uncommenting a single line.

In [ ]:
# Load Protein as PDB file, choose one by uncommenting the rest

# pdb_path = pathlib.Path("data/polyALA.pdb")
# pdb_path = pathlib.Path("data/polyGLY.pdb")
# pdb_path = pathlib.Path("data/polyGV.pdb")
pdb_path = pathlib.Path("data/ligand_2.pdb")
# pdb_path = pathlib.Path("data/ligand_8.pdbqt")
# pdb_path = pathlib.Path("data/ligand_9.pdb")

protein_name = pdb_path.stem
pdb = PDBFile(str(pdb_path))

print("Protein:", protein_name)
print("Atoms:", pdb.topology.getNumAtoms())
print("Residues:", pdb.topology.getNumResidues())

Protein: ligand_2
Atoms: 30
Residues: 1


### Create System

Create the system our simulation will use. We will specify which force fields to use, as well as whether or not our protein will be simulated in a solvent (water) or in a vacuum. 

#### Vacuum

A vacuum simulation means our polypeptide is simulated in isolation, with no surrounding water or ions to iteract with.
This is useful as a "fast physics sandbox" because there are far fewer atoms, so the run is cheaper.

In [ ]:
# Create system (Vacuum, just protein)
box = "vacuum"
forcefield = ForceField("amber14-all.xml")

# Build the final topology/positions we will simulate
modeller = Modeller(pdb.topology, pdb.positions)
modeller.deleteWater()
modeller.addHydrogens(forcefield)

# Create the System from the modeller topology
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=NoCutoff,
    constraints=HBonds
)

# See the size of the system we have to solve, i.e number of equations
print(
    f"Protein: {protein_name} in {box} environment summary:\n{'-'*64}"
    f"\nTotal Number of atoms: {modeller.topology.getNumAtoms()}"
)

u = mda.Universe(modeller)
visualize(u)

ValueError: No template found for residue 0 (LIG).  The residue contains Cl atoms, which are not supported by any template in the force field.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template

#### Solvent (Optional, Unstable on DataHub)

A solvent simulation adds explicit water molecules around the polypeptide. This is closer to the
physical environment of most biomolecules, because water strongly affects stability and dynamics. This may crash your JupyterLab runtime on DataHub. 

In [ ]:
# # Create system (Solvent, protein + water)
# box = "solvent"
# forcefield = ForceField(
#     "amber14-all.xml",
#     "amber14/tip3pfb.xml"
# )

# modeller = Modeller(pdb.topology, pdb.positions)
# modeller.deleteWater()
# modeller.addHydrogens(forcefield)

# # Add an explicit water box (increases atom count a lot)
# modeller.addSolvent(forcefield, padding=1.0*nanometer)

# system = forcefield.createSystem(
#     modeller.topology,
#     nonbondedMethod=PME,
#     nonbondedCutoff=1.0*nanometer,
#     constraints=HBonds,
#     rigidWater=True
# )

# print(
#     f"Protein: {protein_name} in {box} environment summary:\n{'-'*64}"
#     f"\nTotal Number of atoms: {modeller.topology.getNumAtoms()}"
# )

# # u = mda.Universe(modeller)
# # visualize(u, step=10, max_frames=150, water_atoms=1500)

### Experiment Parameters

This cell sets the thermodynamic conditions and the resolution for each MD experiment. The key variable you will change is `temperature`, which sets the thermal energy scale (k_B T) that a thermostat targets by adjusting velocities. Higher temperature increases fluctuations and the likelihood of crossing or falling into energy barriers, while lower temperature reduces both. The remaining parameters control how much trajectory you collect (total simulated time) and how finely you observe it (how often frames are saved).

> You will change the `temperature` for the deliverable assignment.


In [ ]:
# Set parameters for the experiment

temperature = 310 * kelvin # Temperature of the simulation, TODO Change temps to complete the assignment
friction = 1 / picosecond
timestep = 0.002 * picoseconds  # Simulation timestep size

total_time = 100 * picoseconds # Duration of production run

total_steps = int(total_time / timestep) # Total number of steps in production run

traj_record_freq = 500 # How often to record trajectory frames (in steps)

n_frames_saved = total_steps // traj_record_freq  # Estimated number of frames saved

print(
    f"Summary of Molecular Dynamics Simulation Parameters:\n{'-'*64}\n"
    f"Temperature: {temperature}\n"
    f"Friction Coefficient: {friction}\n"
    f"Timestep Size: {timestep}\n"
    f"Total Simulation Time: {total_time}\n"
    f"Total Steps: {total_steps}\n"
    f"Trajectory Recording Frequency: every {traj_record_freq} steps\n"
    f"Estimated Frames Saved: {n_frames_saved}\n"
)


Summary of Molecular Dynamics Simulation Parameters:
----------------------------------------------------------------
Temperature: 310 K
Friction Coefficient: 1 /ps
Timestep Size: 0.002 ps
Total Simulation Time: 100 ps
Total Steps: 50000
Trajectory Recording Frequency: every 500 steps
Estimated Frames Saved: 100



**Rerun note (important):** If you change any of the variables above (especially `temperature`, `timestep`, or `total_time`), rerun the cells in this order:

**Integrator → Simulation → Minimization → (Optional Equilibration) → Production → Visualization**

### Integrator

A Langevin integrator provides temperature control (Thermostat) and performs the time stepping.


In [ ]:
# Set up integrator
integrator = LangevinIntegrator(temperature, friction, timestep)

# For reproducibility
integrator.setRandomNumberSeed(1)

print("Integrator:", type(integrator).__name__)

Integrator: LangevinIntegrator


### Create the Simulation Object 

Run the next cell to build the `Simulation` object and set positions.
If you switch peptide, environment, or integrator settings, you must rerun this cell or create a new object with a different name. 

In [ ]:
simulation = Simulation(
    topology=modeller.topology,
    system=system,
    integrator=integrator
)
simulation.context.setPositions(modeller.positions)

print(f"Molecular Dynamics simulation for {protein_name} in {box} environment ready")

# Initialize velocities to temperature
simulation.context.setVelocitiesToTemperature(temperature)

Molecular Dynamics simulation for polyALA in vacuum environment ready


## Energy Minimization

Energy minimization is a numerical search on the potential energy surface U(x) that removes severe steric clashes and unrealistically large forces in the starting structure. This is not simulating dynamics, and as such it does not represent real time. It simply moves the system to a nearby local minimum so the subsequent MD integration is more stable. We report the potential energy before and after minimization as a basic check: the minimized energy should usually be lower, indicating a more stable structure. 

In [ ]:
# Report the current energy of the system
print(f"Current Energy of the system before minimization:")
energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"Potential Energy: {energy}")

Current Energy of the system before minimization:
Potential Energy: 890.322356402874 kJ/mol


In [ ]:
# Run energy minimization

num_iterations = 100 # Total number of energy minimization iterations
print_iters = 10 # Total number of times to report energy during minimization
substeps = num_iterations // print_iters

print(f"Minimizing energy for {protein_name} in {box} environment...\n{'-'*64}")
for i in range(print_iters):
    simulation.minimizeEnergy(maxIterations=substeps)
    energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    print(f"Iteration {(i+1)*substeps}/{num_iterations}, Potential Energy: {energy}")

Minimizing energy for polyALA in vacuum environment...
----------------------------------------------------------------
Iteration 10/100, Potential Energy: 790.3568598926067 kJ/mol
Iteration 20/100, Potential Energy: 777.0954914987087 kJ/mol
Iteration 30/100, Potential Energy: 769.0312764942646 kJ/mol
Iteration 40/100, Potential Energy: 762.5478611886501 kJ/mol
Iteration 50/100, Potential Energy: 758.9674152731895 kJ/mol
Iteration 60/100, Potential Energy: 755.4845903217793 kJ/mol
Iteration 70/100, Potential Energy: 753.6230865120888 kJ/mol
Iteration 80/100, Potential Energy: 751.2357546389103 kJ/mol
Iteration 90/100, Potential Energy: 749.6844380199909 kJ/mol
Iteration 100/100, Potential Energy: 747.1066602468491 kJ/mol


## Equilibration (Optional)

Equilibration is a short MD run performed before production to reduce issues from initial conditions (fresh velocities, newly added solvent, recently minimized geometry). The goal is to let variables relax and bring the system closer to the target thermodynamic state before you start saving data for analysis. 

In an NVT equilibration, a thermostat controls temperature while volume is fixed. In an NPT equilibration, a thermostat and barostat control temperature and pressure, allowing the box size (and therefore density) to relax, which is mainly relevant for solvated periodic systems.

We do not attach trajectory or log reporters during equilibration so the saved production trajectory reflects the stabilized behavior rather than the initial transient.

In [ ]:
# # Equilibration Parameters

# nvt_duration = 2.0 * picoseconds # Duration of NVT equilibration
# npt_duration = 2.0 * picoseconds # Duration of NPT equilibration
# nvt_timestep = 0.002 * picoseconds # Timestep for NVT equilibration (Same as production)
# npt_timestep = 0.002 * picoseconds # Timestep for NPT equilibration (Same as production)
# nvt_steps = int(round(nvt_duration / nvt_timestep))
# npt_steps = int(round(npt_duration / npt_timestep))

# print(f"Equilibration steps - NVT: {nvt_steps}, NPT: {npt_steps}")

#### NVT

In [ ]:
# simulation.context.setVelocitiesToTemperature(temperature, 1)

# print(f"Running NVT equilibration for {nvt_steps} steps...")
# simulation.step(nvt_steps)
# print("NVT equilibration complete.")

#### NPT

In [ ]:
# barostat = MonteCarloBarostat(1.0*bar, temperature, 25)
# system.addForce(barostat)

# # IMPORTANT: after modifying the System, reinitialize the Context
# simulation.context.reinitialize(preserveState=True)

# print(f"Running NPT equilibration for {npt_steps} steps...")
# simulation.step(npt_steps)
# print("NPT equilibration complete.")

## Production

Production is the primary MD integration phase where the system evolves under the chosen force field and thermodynamic controls. We attach trajectory and log reporters only during production so the saved coordinates and reported quantities correspond to the intended simulation conditions, not the initial relaxation period from initialization, solvation, or minimization. This separation helps ensure that any comparisons you make across conditions reflect the behavior of the model under those conditions rather than equilibration artifacts.

In [ ]:
# Production Run

# Create output directory if it doesn't already exist
# results / box / protein_name / 
output_dir = pathlib.Path("results") / f"{box}" / f"{protein_name}"
output_dir.mkdir(parents=True, exist_ok=True)

# Save topology pdb for loading trajectory onto it later
positions = simulation.context.getState(getPositions=True).getPositions()
topology_pdb_path = output_dir / f"{protein_name}_{box}_topology.pdb"
with open(topology_pdb_path, 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)
print(f"Saved {box} topology PDB to {topology_pdb_path}")



# Start production MD
print(f"Running Production Molecular Dynamics on {protein_name} in {box} environment for {total_steps} steps at {temperature}\n{'-'*64}")

## Create Reporters 
simulation.reporters = []
### Trajectory reporter, records trajectory to DCD file
tempK = int(temperature.value_in_unit(kelvin))
trajectory_path = output_dir / f"{protein_name}_{tempK}K_traj.dcd"
simulation.reporters.append(
    DCDReporter(
        file=str(trajectory_path),
        reportInterval=traj_record_freq,
        enforcePeriodicBox=True
    )
)
### State data reporter, prints to stdout
simulation.reporters.append(
    StateDataReporter(
        stdout,
        total_steps // 10,
        step=True,
        potentialEnergy=True,
        temperature=True,
        density=True,
        volume=True,
        speed=True,
        totalEnergy=True
    )
)
### State data reporter, saves to csv log file
log_path = output_dir / f"{protein_name}_{tempK}K_log.csv"
simulation.reporters.append(
    StateDataReporter(
        str(log_path),
        traj_record_freq,
        step=True,
        potentialEnergy=True,
        temperature=True,
        density=True,
        volume=True,
        speed=True,
        totalEnergy=True,
        separator=","
    )
)
### Checkpoint reporter, saves to binary checkpoint file
checkpoint_path = output_dir / f"{protein_name}_{tempK}K_checkpoint.chk"
simulation.reporters.append(
    CheckpointReporter(
        str(checkpoint_path),
        traj_record_freq
    )
)

print(simulation.context.getPlatform().getName())

# Run the production MD
simulation.step(total_steps)
print("Production MD complete.")
print(f"Trajectory saved to: {trajectory_path}")
print(f"Log saved to: {log_path}")
print(f"Checkpoint saved to: {checkpoint_path}")


# Close reporters to ensure all data is written and files are closed
for reporter in simulation.reporters:
    if hasattr(reporter, 'close'):
        reporter.close()

print("Production MD complete. Reporters closed.")
print(f"Trajectory saved to: {trajectory_path}")

final_pdb_path = output_dir / f"{protein_name}_{tempK}K_final.pdb"
state = simulation.context.getState(getPositions=True)
with open(final_pdb_path, "w") as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)

print(f"Final structure saved to: {final_pdb_path}")


# Load the universe
# Now MDAnalysis will see the correct number of frames because the file is closed
u = mda.Universe(str(topology_pdb_path), str(trajectory_path))

# Visualize
# We use step=1 so we don't skip any of your 50 frames
visualize(u, step=1, max_frames=150, water_atoms=1500)

Saved vacuum topology PDB to results\vacuum\polyALA\polyALA_vacuum_topology.pdb
Running Production Molecular Dynamics on polyALA in vacuum environment for 50000 steps at 310 K
----------------------------------------------------------------
OpenCL
#"Step","Potential Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Box Volume (nm^3)","Density (g/mL)","Speed (ns/day)"
5000,1404.8588003516197,2254.455414250493,324.9061544990243,8.0,0.37258677427526665,--
10000,1492.3853940814734,2241.128191271797,286.3372322396134,8.0,0.37258677427526665,473
15000,1410.279473721981,2307.9730293676257,343.2995804266459,8.0,0.37258677427526665,481
20000,1464.5046734511852,2291.5166767053306,316.2692568520785,8.0,0.37258677427526665,500
25000,1262.5224206447601,2042.6454405002296,298.33778321481924,8.0,0.37258677427526665,507
30000,1329.6634281277657,2146.9882201626897,312.56463457184935,8.0,0.37258677427526665,513
35000,1295.194853901863,2136.574958675541,321.76396402933494,8.0,0.37258677427526

OSError: Reading DCD header failed: premature EOF found in DCD file

Exception ignored in: 'MDAnalysis.lib.formats.libdcd.DCDFile._read_header'
Traceback (most recent call last):
  File "c:\Users\kentc\miniconda3\envs\openmm\lib\site-packages\MDAnalysis\coordinates\DCD.py", line 144, in __init__
    self._file = DCDFile(self.filename)
OSError: Reading DCD header failed: premature EOF found in DCD file


IndexError: index 0 is out of bounds for axis 1 with size 0

Exception ignored in: 'MDAnalysis.lib.formats.libdcd.DCDFile._setup_buffers'
Traceback (most recent call last):
  File "c:\Users\kentc\miniconda3\envs\openmm\lib\site-packages\MDAnalysis\coordinates\DCD.py", line 144, in __init__
    self._file = DCDFile(self.filename)
IndexError: index 0 is out of bounds for axis 1 with size 0


OSError: opened empty file. No frames are saved

In [ ]:
# Plot final structure
u_final = mda.Universe(str(topology_pdb_path), str(final_pdb_path))
print(f"Final Frame of {protein_name} at {tempK}K")
visualize(u_final, step=1, max_frames=1, water_atoms=1500)

Final Frame of polyALA at 310K


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Deliverable Assignment (10 pts)

Answer the following deliverables in a typeset report a submit to bCourses as `lab2_MD_{firstname}_{lastname}.pdf`.  
Your report must be **at least 250 words**.

> You may want to keep your output files in `results/` saved somewhere safe. We may reuse these trajectories later on in the semester.

---

### (10 pts) Temperature Effects in Vacuum (3 Polypeptides)

For each of the three polypeptides provided in `data/`:

- `polyALA.pdb`
- `polyGLY.pdb`
- `polyGV.pdb`

run **vacuum molecular dynamics simulations** at **three different temperatures**:

- **273 K**
- **310 K**
- **373 K**

Keep all other simulation settings the same across runs (timestep, friction, total simulation time).  
Use short runs appropriate for DataHub: **100 ps** per simulation. Take a screenshot of the last frame of each simulation to include in your report for a total of **9 figures**. Include figure captions. 

---

### Report Questions

In your report, address the following:

1. **Folding behavior over time**  
   Describe how each polypeptide’s conformation evolves during the simulation. Comment on whether the chain becomes more compact, remains extended, or exhibits large fluctuations. Note any qualitative differences between early and late frames of the trajectory.

2. **Effect of temperature on final structure**  
   Compare the final conformations at 273 K, 310 K, and 373 K. Discuss how increasing temperature affects structural stability, compactness, and fluctuations. Are higher temperatures associated with partial unfolding or increased disorder?

3. **Comparison across polypeptide sequences**  
   Compare the behavior of `polyALA`, `polyGLY`, and `polyGV` at the same temperature. Which sequences appear more stable or structured in vacuum? Which are more flexible? Relate your observations to the chemical properties of the constituent amino acids (side-chain size and flexibility).

4. **Vacuum simulation limitations**  
   Briefly discuss how simulating these polypeptides in vacuum (without solvent) may influence the observed folding behavior. How might you expect these results to differ if the same simulations were run in an environment of water and ions?

Optionally, you may want to run a simulation with the protein in a solvent to answer and understand these questions. Your mileage may vary with this. 

Your final report should answer all of the following questions and include at least **9 figures**. Submit to bCourses as `lab2_MD_{firstname}_{lastname}.pdf`. 


## Troubleshooting

### Quick checks
If you run into issues, try these first:
1. Restart the notebook kernel, then run again.
2. Confirm your conda environment is active: `conda env list`.
3. Confirm you are in the correct directory: `pwd` and `ls`.
4. Re-run the last command and read the first error message carefully.

### Common issues (MD)
- Simulation is extremely slow:
  Fix: reduce `total_time`, reduce how often you write trajectory frames, and close extra browser tabs/apps.
- "FileNotFoundError" when loading a `.pdb`:
  Fix: confirm the file is in `data/` and your path matches (for example, `data/{protein_file}.pdb`).
- "ModuleNotFoundError" for OpenMM or MDAnalysis:
  Fix: confirm the correct kernel is selected, then recreate the environment from `environment.yml`.
- Simulation "blows up" (NaNs, huge energies):
  Fix: re-run minimization, reduce the timestep, and confirm constraints are enabled.
- Trajectory file not created in `results/`:
  Fix: run the simulation cell again and confirm you have write permissions in the lab folder.
- Embedded videos do not display (`.mp4` missing):
  Fix: open the `results/` folder to confirm the file exists. If it does, try restarting the kernel and re-running the cell. If you are running locally, you may need `ffmpeg` installed to generate videos (or switch the helper code in `md1.py` to save as a `.gif` instead).


## Collaboration and Academic Integrity

You may discuss high-level ideas and debugging strategies with classmates.
Do not share completed code, notebooks, or written answers.
All submitted work must be your own.

Cite any external resources used (links are fine).
If you received help, include a short acknowledgment in your submission.

## References

Lab links used in this document:
- OpenMM - simulation engine and docs - https://openmm.org/
- MDAnalysis - trajectory loading and analysis - https://www.mdanalysis.org/
- nglview - interactive molecular visualization in Jupyter - https://nglviewer.org/nglview/latest/
- RCSB Protein Data Bank - protein structure download - https://www.rcsb.org/
- Molecular dynamics background - https://en.wikipedia.org/wiki/Molecular_dynamics
- Numerical methods for ODEs (background) - https://en.wikipedia.org/wiki/Numerical_methods_for_ordinary_differential_equations
- Protein structure background - https://en.wikipedia.org/wiki/Protein_structure

Sources used to create or adapt this lab:
- OpenMM documentation and examples - https://openmm.org/
